# Data Pipeline Demo

This notebook demonstrates the current citation-network data pipeline:

- load a Planetoid dataset
- create a link-prediction split
- save the split artifact
- inspect the saved tensors


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

PosixPath('/IdeaProjects/citegnn')

In [2]:
import torch

from citegnn.data import load_citation_graph, create_link_prediction_split, save_split_artifact

In [3]:
dataset_name = "Cora"

graph = load_citation_graph(dataset_name)
split = create_link_prediction_split(graph, seed=42)
artifact_path = save_split_artifact(split, output_dir=PROJECT_ROOT / "artifacts" / "splits")

summary = {
    "dataset": graph.name,
    "num_nodes": graph.num_nodes,
    "num_features": graph.num_features,
    "train_nodes": int(graph.train_mask.sum()),
    "val_nodes": int(graph.val_mask.sum()),
    "test_nodes": int(graph.test_mask.sum()),
    "input_graph_edge_count": int(graph.edge_index.size(1)),
    "train_graph_edge_count": int(split.train_graph_edge_index.size(1)),
    "val_graph_edge_count": int(split.val_graph_edge_index.size(1)),
    "test_graph_edge_count": int(split.test_graph_edge_index.size(1)),
    "train_pos_edges": int(split.train_pos_edge_index.size(1)),
    "val_pos_edges": int(split.val_pos_edge_index.size(1)),
    "test_pos_edges": int(split.test_pos_edge_index.size(1)),
    "train_neg_edges": int(split.train_neg_edge_index.size(1)),
    "val_neg_edges": int(split.val_neg_edge_index.size(1)),
    "test_neg_edges": int(split.test_neg_edge_index.size(1)),
    "artifact_path": str(artifact_path.relative_to(PROJECT_ROOT)),
}

summary

Processing...
/IdeaProjects/citegnn/.venv/lib/python3.14/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/IdeaProjects/citegnn/.venv/lib/python3.14/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/IdeaProjects/citegnn/.venv/lib/python3.14/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/IdeaProjects/citegnn/.ven

{'dataset': 'Cora',
 'num_nodes': 2708,
 'num_features': 1433,
 'train_nodes': 140,
 'val_nodes': 500,
 'test_nodes': 1000,
 'input_graph_edge_count': 10556,
 'train_graph_edge_count': 7392,
 'val_graph_edge_count': 7392,
 'test_graph_edge_count': 8446,
 'train_pos_edges': 3696,
 'val_pos_edges': 527,
 'test_pos_edges': 1055,
 'train_neg_edges': 3696,
 'val_neg_edges': 527,
 'test_neg_edges': 1055,
 'artifact_path': 'artifacts/splits/cora_seed42_lp.pt'}

In [4]:
artifact = torch.load(artifact_path, map_location="cpu")

{key: tuple(value.shape) if hasattr(value, "shape") else value for key, value in artifact.items()}

{'dataset_name': 'Cora',
 'is_undirected': True,
 'seed': 42,
 'val_ratio': 0.1,
 'test_ratio': 0.2,
 'negative_ratio': 1.0,
 'train_mask': (2708,),
 'val_mask': (2708,),
 'test_mask': (2708,),
 'train_graph_edge_index': (2, 7392),
 'val_graph_edge_index': (2, 7392),
 'test_graph_edge_index': (2, 8446),
 'train_pos_edge_index': (2, 3696),
 'val_pos_edge_index': (2, 527),
 'test_pos_edge_index': (2, 1055),
 'train_neg_edge_index': (2, 3696),
 'val_neg_edge_index': (2, 527),
 'test_neg_edge_index': (2, 1055)}

To switch datasets, change `dataset_name` to `"PubMed"` and rerun the notebook.